# Colab training — Akkadian → English (ByT5)

Тонкая обёртка: клонирует `ml-dev`, ставит пакет, готовит данные, запускает обучение через CLI.

**Перед запуском** добавь в Colab Secrets (ключик слева):
- `WANDB_API_KEY` — wandb.ai → Settings → API keys
- `KAGGLE_USERNAME`, `KAGGLE_KEY` — kaggle.com → Settings → Create New Token
- `HF_TOKEN` — (опционально) для push_to_hub

Runtime → Change runtime type → **T4 GPU**. При обрыве сессии просто перезапусти все ячейки — обучение продолжится с последнего чекпоинта в Drive.

In [ ]:
CONFIG = "configs/exp2_full_seed13.yaml"  # <- какой эксперимент запускаем
BRANCH = "ml-dev"

!nvidia-smi -L

In [ ]:
# Drive — чекпоинты переживают обрыв сессии
from google.colab import drive
drive.mount("/content/drive")
import os
PERSIST = "/content/drive/MyDrive/akkadian_nmt"
os.makedirs(PERSIST, exist_ok=True)

In [ ]:
!git clone --branch {BRANCH} https://github.com/ObjoradDdd/ml-hits-3-lab.git /content/repo
%cd /content/repo/ml
!pip install -q -e . sacrebleu
# модельные чекпоинты пишем в Drive
!ln -sfn {PERSIST} /content/repo/ml/model

In [ ]:
# секреты -> окружение
from google.colab import userdata
for key in ("WANDB_API_KEY", "KAGGLE_USERNAME", "KAGGLE_KEY", "HF_TOKEN"):
    try:
        os.environ[key] = userdata.get(key)
    except Exception:
        print("no secret:", key)

In [ ]:
# данные соревнования
!pip install -q kaggle
!kaggle competitions download -c deep-past-initiative-machine-translation -p data/
!cd data && unzip -oq deep-past-initiative-machine-translation.zip && rm -f publications.csv  # 580MB, не нужен
!python -m akkadian_nmt.data_prep --data_dir=./data --out_dir=./data/processed

In [ ]:
# обучение (возобновляется с последнего чекпоинта автоматически)
!python -m akkadian_nmt.train --config={CONFIG}

In [ ]:
# оценка на dev: greedy vs beam {1,4,8}
import yaml
cfg = yaml.safe_load(open(CONFIG))
final = cfg["output_dir"] + "/final"
!python -m akkadian_nmt.evaluate beam_sweep --model_dirs={final} --max_samples=200

In [ ]:
# полный метрический набор (BLEU + chrF++ + geo-mean + COMET) на лучшей конфигурации
!pip install -q unbabel-comet
!python -m akkadian_nmt.evaluate run --model_dirs={final} --num_beams=4 --comet=True \
    --out_file=data/dev_predictions.json

In [ ]:
# сабмит на Kaggle
!python model.py predict-file --dataset=data/test.csv --model_dir={final}
from google.colab import files
files.download("data/results.csv")  # -> загрузить на Kaggle вручную
# либо: !kaggle competitions submit -c deep-past-initiative-machine-translation -f data/results.csv -m "run"